# Stream-CQSA tutorial: usage, profiling, and OOM recovery

Three things, in order:

1. **Using it** — the API, and how it differs from `scaled_dot_product_attention`.
2. **Profiling it** — measuring memory and time so the numbers mean something.
3. **Comparing it under a memory cap** — capping the GPU to 1/2/4 GiB brings the
   OOM boundary down to a sequence length that runs in seconds, so the whole
   comparison fits in a notebook instead of a 20-hour cluster job.

That third idea is what makes this practical. Attention memory is linear in `N`,
so *capping memory* and *raising N* are interchangeable ways to reach the same
boundary — and one of them is free.

Runs in a couple of minutes on any A100.

In [ ]:
import gc, os, sys, time, warnings
import torch
import torch.nn.functional as F

def _find_pkg():
    here = os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd()))
    if os.path.isfile(here): here = os.path.dirname(here)
    for base in (here, os.getcwd()):
        d = base
        for _ in range(6):
            c = os.path.join(d, "packages", "stream-cqsa")
            if os.path.isfile(os.path.join(c, "stream_cqsa", "stable_stream.py")): return c
            if os.path.dirname(d) == d: break
            d = os.path.dirname(d)
    return None

PKG = _find_pkg()
if PKG is None:
    raise RuntimeError("run this from inside the Stream-CQSA-dev tree")
for _m in [m for m in sys.modules if m == "stream_cqsa" or m.startswith("stream_cqsa.")]:
    del sys.modules[_m]
sys.path.insert(0, PKG); warnings.filterwarnings("ignore")

from stream_cqsa.stable_stream import stream_cqsa_forward, stream_cqsa_backward, TraceRecorder
import cqsa_cuda  # noqa: F401
TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU {torch.cuda.get_device_name(0)}  ({TOTAL_GIB:.0f} GiB)")
print(f"package: {PKG}")

## 1. Using it

Stream-CQSA is **not** a drop-in for `scaled_dot_product_attention`. Three
differences, all deliberate:

| | SDPA | Stream-CQSA |
|---|---|---|
| returns | a tensor | `(out, info)` |
| output dtype | input dtype | **fp32** |
| autograd | yes | no — call the backward explicitly |

`info["lse"]` is the global log-sum-exp. It is the piece that makes the
decomposition exact, and the backward requires it.

In [ ]:
B, H, N, D = 1, 4, 8192, 64
CAUSAL, SCALE = True, 64 ** -0.5
torch.manual_seed(0)
q, k, v = (torch.randn(B, H, N, D, device="cuda", dtype=torch.float16) for _ in range(3))

out, info = stream_cqsa_forward(q, k, v, itr=1, causal=CAUSAL)

print(f"out       {tuple(out.shape)}  {out.dtype}")
print(f"lse       {tuple(info['lse'].shape)}  {info['lse'].dtype}")
for key in ("itr", "n_subproblems", "n_parallel", "untouched_tokens", "plan_reason"):
    if key in info: print(f"{key:<18}{info[key]}")

ref = F.scaled_dot_product_attention(q, k, v, is_causal=CAUSAL, scale=SCALE)
print(f"\nvs SDPA: rel err {float((out - ref.float()).norm() / ref.float().norm()):.3e}")
print("untouched_tokens must be 0 -- non-zero would mean the decomposition "
      "missed tokens, i.e. a coverage bug")

### `itr` — the decomposition depth

`itr=0` means *no* decomposition (one monolithic call). `itr=1` splits into 7
subproblems, `itr=2` into 49. Deeper means smaller pieces and less peak memory,
at the cost of more total work.

`itr="auto"` asks the planner to choose. **It returns 0 whenever a monolithic
call fits** — below the OOM boundary, decomposing is strictly worse.

In [ ]:
for itr in (0, 1, 2, "auto"):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    o, i = stream_cqsa_forward(q, k, v, itr=itr, causal=CAUSAL)
    wk = (torch.cuda.max_memory_allocated() - base) / 2**20
    print(f"itr={str(itr):<5} -> ran at itr={i['itr']}, {i['n_subproblems']:>2} subproblems, "
          f"workspace {wk:6.1f} MiB")
    del o, i

### Backward

Pass `info["lse"]` through from the forward. The output must be cast back to the
input dtype (the forward returns fp32).

In [ ]:
dout = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
out, info = stream_cqsa_forward(q, k, v, itr=1, causal=CAUSAL)
dq, dk, dv = stream_cqsa_backward(q, k, v, dout, out.to(q.dtype), info["lse"],
                                  itr=1, causal=CAUSAL)

qq, kk, vv = (t.detach().requires_grad_() for t in (q, k, v))
o = F.scaled_dot_product_attention(qq, kk, vv, is_causal=CAUSAL, scale=SCALE)
rq, rk, rv = torch.autograd.grad(o, [qq, kk, vv], dout)
r = lambda a, b: float((a.double() - b.double()).norm() / b.double().norm())
print(f"dQ {r(dq,rq):.3e}   dK {r(dk,rk):.3e}   dV {r(dv,rv):.3e}   (vs SDPA autograd)")

### Host streaming — the OOM-recovery mode

`stream_from_host=True` with **CPU** tensors keeps Q/K/V in host memory and moves
only one subsequence at a time to the GPU. This is the configuration that
survives past the point where the baselines fail: device-resident Stream-CQSA
still needs the whole input set on the GPU, so it OOMs alongside them.

In [ ]:
qc, kc, vc = (t.cpu() for t in (q, k, v))
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
o_h, i_h = stream_cqsa_forward(qc, kc, vc, itr=1, causal=CAUSAL, stream_from_host=True)
print(f"streamed: peak GPU {torch.cuda.max_memory_allocated()/2**20:6.1f} MiB, "
      f"itr={i_h['itr']}, out on {o_h.device}")
print(f"matches device-resident run: "
      f"{float((o_h.cuda()-out).norm()/out.norm()):.3e}")

## 2. Profiling it properly

Four things that will otherwise give you wrong numbers.

**Drain the allocator between measurements.** Otherwise a previous method's
cached blocks either inflate or mask the next one.

**Warm up.** The first call of any method pays CUDA context and kernel load. In
our own sweep this inflated one FlashAttention measurement by **5629×**
(12,090 ms against a true 2.1 ms). One warmup call is not always enough.

**Split memory into inputs + workspace.** Peak alone is not comparable across
methods that hold their inputs in different places: a host-streaming method's
peak falls partly because Q/K/V are elsewhere, not because attention got leaner.

**Use median of several reps**, not a single shot.

In [ ]:
def profile(fn, *, warmup=2, reps=3, inputs=()):
    """Time + memory for one attention call. Returns a dict."""
    for _ in range(warmup):
        r = fn(); del r
    gc.collect(); torch.cuda.synchronize(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()          # AFTER inputs exist
    ts = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        r = fn()
        torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1e3)
    peak = torch.cuda.max_memory_allocated()
    seen, dev_in = set(), 0
    for t in inputs:
        if torch.is_tensor(t) and t.is_cuda and t.data_ptr() not in seen:
            seen.add(t.data_ptr()); dev_in += t.numel() * t.element_size()
    ts.sort()
    del r
    return dict(ms=ts[len(ts)//2],
                workspace_mib=(peak - base) / 2**20,
                inputs_mib=dev_in / 2**20,
                total_mib=(dev_in + peak - base) / 2**20)

p_sdpa = profile(lambda: F.scaled_dot_product_attention(q,k,v,is_causal=CAUSAL,scale=SCALE),
                 inputs=(q,k,v))
p_cqsa = profile(lambda: stream_cqsa_forward(q,k,v,itr=1,causal=CAUSAL)[0], inputs=(q,k,v))
print(f"{'method':<16}{'ms':>9}{'inputs':>10}{'workspace':>11}{'total MiB':>11}")
for nm, p in (("SDPA", p_sdpa), ("Stream-CQSA", p_cqsa)):
    print(f"{nm:<16}{p['ms']:9.2f}{p['inputs_mib']:10.0f}{p['workspace_mib']:11.0f}{p['total_mib']:11.0f}")

### Where the time goes

`TraceRecorder` breaks a Stream-CQSA run into stages. **Caveat:** with several
subproblems in flight, a CUDA event span also covers time queued behind other
streams, so the stage totals over-count wall time. Use `max_parallel=1` when you
want them to be attributable.

In [ ]:
tr = TraceRecorder(enabled=True)
stream_cqsa_forward(q, k, v, itr=2, causal=CAUSAL, trace=tr, max_parallel=1)
tot = tr.stage_totals_ms()
print("attributable stage totals (max_parallel=1):")
for kk, vv in tot.items(): print(f"   {kk:>9} {vv:8.2f} ms")

## 3. Comparing under a memory cap

`torch.cuda.set_per_process_memory_fraction(f)` limits what this process may
allocate. Because attention memory is **linear in N**, halving the budget is
equivalent to doubling the sequence length — so a cap reaches the same OOM
boundary as a huge `N`, in seconds rather than hours.

The helper below runs one method under a given cap and reports completed / OOM.

### The Stream-CQSA variants, and what each one moves

**The rows below are diagnostic configurations, not recommendations.** They fix
`itr` and residency by hand so you can see what each knob does. For real use,
skip to `stream_cqsa_auto` at the end of this section — it picks these for you.

Device memory during a run is the sum of **four independent O(N) terms**, and
each variant removes or shrinks a different one:

| term | at `N=1M, H=8, D=64, fp16` | controlled by |
|---|--:|---|
| **inputs** — Q/K/V resident on the GPU | 3.0 GiB | `stream_from_host=True` |
| **accumulator** — `acc` fp32 `[B,N,H,D]` plus `l`, `m` | 2.0 GiB | `accumulate_on_gpu=False` |
| **output** — returned fp32 `[B,N,H,D]` | 2.0 GiB | *(currently always on the device)* |
| **in-flight subsequences** | ~1.3 GiB at `itr=1`, ~0.55 at `itr=2` | deeper `itr` |

**This is the point of the whole section: `itr` only shrinks the last term.**
The other three are invariant in `itr`, so "just increase `itr`" does *not* always
avoid an OOM — it stalls at a floor set by the inputs, the accumulator and the
output. That floor is exactly why the `streamed` row below still fails under a
tight cap while `min-device` does not: `streamed` leaves the 2.0 GiB accumulator
on the GPU, and no depth of decomposition touches it.

The four rows are a ladder over *which terms stay resident*:

| variant | streaming? | itr | inputs | accum | in one line |
|---|---|---|---|---|---|
| `itr=1` | **no** | fixed 1 | GPU | GPU | decompose only — still holds everything, **plus** an accumulator |
| `itr=2` | **no** | fixed 2 | GPU | GPU | finer pieces; the resident terms are untouched |
| `streamed (auto itr)` | yes | **automatic** | host | GPU | removes the largest single term |
| `min-device (auto itr)` | yes | **automatic** | host | host | GPU holds only work in flight (and the output) |

**Depth is automatic by default.** `stream_cqsa_forward` now defaults to
`itr="auto"`, and the two streaming rows below escalate the depth on OOM rather
than pinning it — that is how they are meant to be used. Pinning `itr` is the
explicit opt-out, for reproducing a specific decomposition.

The first two rows are **not** streaming between CPU and GPU, and their `itr` is
pinned **on purpose** — they are there to demonstrate the failure mode that
motivates automatic escalation. Because they still carry ~5.0 GiB of resident
terms against a baseline's 3.0 GiB, **decomposing alone makes memory worse**, and
they OOM *earlier* than SDPA. That is not a defect; it is the reason `itr` must
be chosen together with residency rather than on its own.

In [ ]:
# The three terms, measured. Small N so this runs in seconds; the ratios are
# what matter, and they hold at any N (all three terms are linear in N).
Nq, Bq, Hq, Dq = 65536, 1, 8, 64
print(f"N={Nq}: Q/K/V would be {3*Nq*Hq*Dq*2/2**20:.0f} MiB on the GPU, "
      f"fp32 accumulator {Nq*Hq*Dq*4/2**20:.0f} MiB\n")

VARIANTS = [
    ("itr=1",      dict(itr=1)),
    ("itr=2",      dict(itr=2)),
    ("streamed",   dict(itr=2, stream_from_host=True)),
    ("min-device", dict(itr=2, stream_from_host=True, accumulate_on_gpu=False)),
]
print(f"{'variant':<12}{'peak GPU MiB':>14}{'wall ms':>10}   inputs / accumulator on")
for name, kw in VARIANTS:
    host = kw.get("stream_from_host", False)
    # Build the inputs ON the device they belong on. Creating them on the GPU and
    # calling .cpu() would COPY: the GPU original stays alive behind our own
    # reference and keeps counting toward the peak, which would report streaming
    # as saving nothing.
    g = torch.Generator().manual_seed(0)
    mk = lambda: torch.randn(Bq, Hq, Nq, Dq, generator=g, dtype=torch.float16)
    a, b, c = mk(), mk(), mk()
    if not host:
        a, b, c = a.cuda(), b.cuda(), c.cuda()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    o, i = stream_cqsa_forward(a, b, c, causal=True, **kw)
    torch.cuda.synchronize()
    where = ("host" if host else "GPU") + " / " + \
            ("host" if kw.get("accumulate_on_gpu", True) is False else "GPU")
    print(f"{name:<12}{torch.cuda.max_memory_allocated()/2**20:14.0f}"
          f"{(time.perf_counter()-t0)*1e3:10.1f}   {where}")
    del o, i, a, b, c
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
def under_cap(cap_gib, fn_builder, *, N=262144, B=1, H=8, D=64, dtype=torch.float16,
              host=False):
    """Allocate inputs and run one method with the process capped to cap_gib."""
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(min(cap_gib / TOTAL_GIB, 1.0), 0)
    rec = dict(cap=cap_gib, status="ok", ms=float("nan"), peak_mib=float("nan"), itr=None)
    try:
        # Generate on the HOST, then place. Generating on the GPU would burn
        # device budget (and an fp32 intermediate) before the method even runs,
        # so a tight cap would report "OOM" for the generator rather than for
        # the method under test -- which is not what we are measuring.
        g = torch.Generator().manual_seed(0)
        mk = lambda: torch.randn(B, H, N, D, generator=g, dtype=torch.float32).to(dtype)
        a, b, c = mk(), mk(), mk()
        if not host:
            a, b, c = a.cuda(), b.cuda(), c.cuda()
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        res = fn_builder(a, b, c)
        torch.cuda.synchronize()
        rec["ms"] = (time.perf_counter() - t0) * 1e3
        rec["peak_mib"] = torch.cuda.max_memory_allocated() / 2**20
        if isinstance(res, tuple): rec["itr"] = res[1].get("itr")
        del a, b, c, res
    except torch.cuda.OutOfMemoryError:
        rec["status"] = "OOM"
    except RuntimeError as e:
        rec["status"] = "OOM" if "out of memory" in str(e).lower() else "error"
    finally:
        gc.collect(); torch.cuda.empty_cache()
        torch.cuda.set_per_process_memory_fraction(1.0, 0)   # always reset
    return rec

N_DEMO = 1048576
# Escalate the depth until it fits, holding residency fixed. NOTE this is
# escalate-on-OOM, not itr="auto": the planner sizes itself from device-wide
# free memory and cannot see a per-process cap, so under a cap it would pick a
# depth that does not fit. Escalation observes the actual failure instead.
def auto_itr(residency_kw, max_itr=4):
    def run(a, b, c):
        for depth in range(1, max_itr + 1):
            try:
                return stream_cqsa_forward(a, b, c, itr=depth, causal=True, **residency_kw)
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if "out of memory" not in str(e).lower(): raise
                gc.collect(); torch.cuda.empty_cache()
        raise torch.cuda.OutOfMemoryError(f"no fit up to itr={max_itr}")
    return run

METHODS = [
    ("SDPA",                  lambda a,b,c: F.scaled_dot_product_attention(a,b,c,is_causal=True,scale=SCALE), False),
    ("FlashAttention-2",      lambda a,b,c: __import__("flash_attn").flash_attn_func(
                                  a.transpose(1,2),b.transpose(1,2),c.transpose(1,2),
                                  causal=True,softmax_scale=SCALE), False),
    # FIXED depth, NO streaming -- kept fixed on purpose, as the failure example.
    ("CQSA itr=1 (fixed)",    lambda a,b,c: stream_cqsa_forward(a,b,c,itr=1,causal=True), False),
    ("CQSA itr=2 (fixed)",    lambda a,b,c: stream_cqsa_forward(a,b,c,itr=2,causal=True), False),
    # AUTOMATIC depth -- how these are meant to be used.
    ("streamed (auto itr)",   auto_itr(dict(stream_from_host=True)), True),
    # low_memory=True also moves the fp32 accumulator to the host. That
    # accumulator is [B,N,H,D] fp32 and sits on the device by default, so it is
    # a floor no depth of decomposition can get under -- which is exactly why
    # automatic depth alone is not enough and residency has to escalate too.
    ("min-device (auto itr)", auto_itr(dict(stream_from_host=True, low_memory=True)), True),
]
CAPS = [10.0, 6.0, 4.5, 3.5, 2.5]
# ~15 min. N is chosen so the inputs (3 GiB) dominate: that is what makes the
# methods separate cleanly across the cap ladder. At N=262144 the inputs are only
# 0.75 GiB, the baselines are already frugal, and everything dies within one cap
# step of everything else.
print(f"forward, N={N_DEMO}, fp16, causal   (inputs alone = "
      f"{3*N_DEMO*8*64*2/2**30:.2f} GiB)\n")
print("fixed-itr rows are the failure example; auto-itr rows show the settled depth (i1/i2/...)\n")
print(f"{'method':<24}" + "".join(f"{c:>10.1f}G" for c in CAPS))
print("-" * (24 + 11*len(CAPS)))
for name, fn, host in METHODS:
    cells = []
    for cap in CAPS:
        r = under_cap(cap, fn, N=N_DEMO, host=host)
        cells.append((f"{r['ms']/1000:6.1f}s/i{r['itr']}" if r.get("itr") is not None
                      else f"{r['ms']/1000:9.2f}s") if r["status"]=="ok" else f"{'OOM':>10}")
    print(f"{name:<24}" + "".join(cells))

### Reading it

Measured output of the cell above (A100, N=1048576, fp16, causal):

| method | 10.0G | 6.0G | 4.5G | 3.5G | 2.5G |
|---|--:|--:|--:|--:|--:|
| SDPA | 8.6 s | 8.7 s | 8.7 s | OOM | OOM |
| FlashAttention-2 | 12.8 s | 8.7 s | 8.7 s | OOM | OOM |
| Stream-CQSA itr=1 | OOM | OOM | OOM | OOM | OOM |
| Stream-CQSA itr=2 | 22.9 s | OOM | OOM | OOM | OOM |
| Stream-CQSA streamed | 17.5 s | 18.1 s | 22.6 s | OOM | OOM |
| **Stream-CQSA min-device** | 47.1 s | 45.6 s | 45.9 s | **45.1 s** | **46.4 s** |

Read it right to left, as a *minimum budget* per method.

* The baselines are fast but stop at 4.5 GiB: they need Q/K/V (3 GiB) plus the
  output resident, and nothing about them bends.
* **Device-resident Stream-CQSA is worse than the baselines**, and `itr=1` is
  worse than `itr=2`. That is not a bug. It still holds the whole input set on
  the GPU *and* adds an accumulator, so decomposing alone buys nothing.
* Streaming removes the input residency and buys one more cap step.
* Only `min-device` -- streamed **and** with the fp32 accumulator on the host --
  clears every rung, at roughly 5x the time of SDPA. That is the trade in one
  line: it is the slowest column and the only one with an entry at 2.5 GiB.

The escalation cell shows the guardrail directly: at a 3.5 GiB cap it settles at
`itr=1`, at 2.5 GiB it goes deeper to `itr=2`. The decomposition depth rises as
the budget tightens, which is the whole mechanism.

### Why `streamed` still OOMs, at *any* depth

This is the most useful row in the table, so it is worth doing the arithmetic.

`streamed` puts the inputs on the host but leaves the **fp32 accumulator** on the
GPU. Add the fp32 **output**, which is currently always returned to the device,
and you get a floor that no value of `itr` can get under:

| term (N=1M, H=8, D=64) | `streamed` | `min-device` | shrinks with `itr`? |
|---|--:|--:|---|
| inputs Q/K/V | 0 (host) | 0 (host) | no |
| fp32 accumulator `[B,N,H,D]` | **2.0 GiB** | 0 (host) | **no** |
| fp32 output `[B,N,H,D]` | 2.0 GiB | 2.0 GiB | **no** |
| in-flight subsequences | 1.3 GiB `itr=1` → 0.55 `itr=2` → … | same | yes |
| **floor, before any work** | **4.0 GiB** | **2.0 GiB** | — |

So under a 3.5 GiB cap `streamed` is already over budget from two tensors alone,
and escalating `itr` shrinks only the one term that was never the problem. The
escalation loop dutifully tries `itr=1,2,3,4`, fails at each, and gives up —
which is exactly what the table shows.

`min-device` removes the accumulator, dropping the floor to 2.0 GiB, and then
`itr` has something to work with. That is why it settles at a *shallow* depth
(`itr=1`) even at the tightest cap: once the O(N) resident terms are gone, the
subsequences were never the binding constraint.

**The general lesson:** when a run OOMs, first ask *which* term is binding. If it
is an O(N) resident term — inputs, accumulator, output — then decomposing harder
cannot help, and you need to move something off the device instead. `itr` is the
wrong knob for three of the four terms.

*(This also explains the remaining gap to an unconditional guarantee: the output
term is not yet relocatable. At N=262144 it is 512 MiB, which is half of a
1.0 GiB budget before anything runs.)*

In [ ]:
# The floor, computed. Nothing here runs attention -- it is just the resident
# terms, which is the point: they are known before you start.
def floors(N, B=1, H=8, D=64, itemsize=2, c=7, l=3):
    acc = B*N*H*D*4 / 2**20          # fp32 accumulator
    out = B*N*H*D*4 / 2**20          # fp32 output, returned to the device
    inp = 3*B*N*H*D*itemsize / 2**20 # Q/K/V
    sub = lambda itr: 3*B*int(N*(l/c)**itr)*H*D*itemsize / 2**20
    print(f"N={N}  (MiB)")
    print(f"  {'variant':<22}{'inputs':>9}{'accum':>9}{'output':>9}{'floor':>9}"
          f"{'+itr=1':>9}{'+itr=2':>9}")
    for name, i_dev, a_dev in (("device-resident", True,  True),
                               ("streamed",        False, True),
                               ("min-device",      False, False)):
        fi, fa = (inp if i_dev else 0), (acc if a_dev else 0)
        fl = fi + fa + out
        print(f"  {name:<22}{fi:9.0f}{fa:9.0f}{out:9.0f}{fl:9.0f}"
              f"{fl+sub(1):9.0f}{fl+sub(2):9.0f}")

floors(262144)
print()
floors(1048576)
print("\nCompare the 'floor' column with the caps in the table above: a variant whose"
      "\nfloor already exceeds the cap cannot be rescued by any itr.")

In [ ]:
# The escalation pattern, for when you cannot choose itr up front.
# Note stream_from_host=True: under a tight cap the device-resident path cannot
# fit at ANY depth once the inputs alone exceed the budget, so escalating itr
# only helps if the inputs are elsewhere.
def cqsa_escalating(a, b, c, max_itr=4, **kw):
    """Deepen the decomposition until it fits -- the guardrail behaviour."""
    for depth in range(1, max_itr + 1):
        try:
            return stream_cqsa_forward(a, b, c, itr=depth, causal=True, **kw)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower(): raise
            gc.collect(); torch.cuda.empty_cache()
    raise torch.cuda.OutOfMemoryError(f"did not fit even at itr={max_itr}")

for cap in (3.5, 2.5):
    r = under_cap(cap, lambda a,b,c: cqsa_escalating(a,b,c,stream_from_host=True,accumulate_on_gpu=False),
                  N=N_DEMO, host=True)
    print(f"escalating, streamed, {cap} GiB cap: {r['status']:>3}  "
          f"settled at itr={r['itr']}  peak {r['peak_mib']:.0f} MiB")

## The recommended way to call it

Everything above is diagnostic. In practice use `stream_cqsa_auto`, which walks a
cheapest-first ladder and returns the first configuration that fits — escalating
`itr` **and** residency, because as shown above `itr` alone stalls at a floor.

```python
from stream_cqsa.oom_fallback import stream_cqsa_auto
out = stream_cqsa_auto(q, k, v, causal=True)            # just runs
out, info = stream_cqsa_auto(q, k, v, causal=True, return_info=True)
info["config"]        # the rung it settled on
info["rungs_tried"]   # the ones that OOMed first
info["lse"]           # pass to stream_cqsa_backward
```

Measured, N=262144 (inputs 768 MiB, accumulator 512 MiB), forward:

| GPU budget | settles at | peak |
|---|---|--:|
| 4.0 GiB | `itr=1` | 2429 MiB |
| 2.0 GiB | `itr=2` | 1986 MiB |
| 1.0 GiB | `itr=2` + streamed + host accumulator | 919 MiB |

It escalates exactly as intended and needs **no configuration from the caller**.

### An honest limit

It is not yet an unconditional guarantee. Below ~1.0 GiB at this `N` it still
fails, and the reason is the **output**: the forward ends with
`return out.to(device)`, so the fp32 `[B,N,H,D]` result is moved to the GPU even
when the accumulator lives on the host. At N=262144 that is 512 MiB — 67% of a
0.75 GiB budget — before any work is done. Keeping the output host-resident when
the inputs are is the remaining change needed for "always executes" to be literally
true; until then, treat the ladder as *very* hard to exhaust rather than
impossible.

## Summary

* `stream_cqsa_forward` returns `(out, info)` in fp32; keep `info["lse"]` for the
  backward, which is a separate explicit call.
* Device memory is four O(N) terms; **`itr` shrinks only one of them**, so depth
  alone cannot always avoid an OOM.
* `stream_from_host=True` moves the inputs off; `accumulate_on_gpu=False` moves
  the accumulator off. Together they are what clears the tightest budgets.
* `stream_cqsa_auto` combines all of this and is the recommended entry point.
* Profile with the allocator drained, after warmup, splitting inputs from
  workspace, over several reps.
* Capping GPU memory stands in for a long sequence — memory is linear in `N`.
  Pass an explicit `itr` under a cap, because the planner reads device-wide free
  memory and cannot see the cap.

For the measured comparison against SDPA and FlashAttention out to N=16M see
`docs/REPORT.md` §8-9; for implementation technique, `docs/METHODOLOGY.md`.

In [ ]:
from stream_cqsa.oom_fallback import stream_cqsa_auto

# The no-configuration path: hand it Q/K/V and it finds a rung that fits.
for cap in (4.0, 2.0, 1.0):
    torch.cuda.set_per_process_memory_fraction(min(cap / TOTAL_GIB, 1.0), 0)
    try:
        g = torch.Generator().manual_seed(0)
        mk = lambda: torch.randn(1, 8, 262144, 64, generator=g, dtype=torch.float16)
        a, b, c = mk().cuda(), mk().cuda(), mk().cuda()
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        out, info = stream_cqsa_auto(a, b, c, causal=True, return_info=True)
        print(f"{cap:4.1f} GiB budget -> settled at {info['config']}, "
              f"peak {torch.cuda.max_memory_allocated()/2**20:.0f} MiB, "
              f"{len(info['rungs_tried'])} rung(s) OOMed first")
        del a, b, c, out, info
    finally:
        gc.collect(); torch.cuda.empty_cache()
        torch.cuda.set_per_process_memory_fraction(1.0, 0)